# Day 12: Week 2 综合复习 —— 知识速查 + 综合场景演练

> **目标**: 回顾 Week 2 六天核心内容(SQL子查询/JOIN + Python函数/异常/文件IO + NumPy)，通过综合场景串联所有知识点。
> **建议用时**: 30 分钟回顾 + 60 分钟习题

## 1. SQL 速查卡

### 子查询四种位置

| 位置 | 形态 | 用途 | 示例 |
|------|------|------|------|
| WHERE 比较 | 标量子查询(1行1列) | 把聚合值算出来再比较 | `WHERE total > (SELECT AVG(total) FROM sales)` |
| WHERE IN | IN子查询(1列多行) | 先圈出一批key，再捞明细 | `WHERE id IN (SELECT customer_id FROM ...)` |
| WHERE EXISTS | 相关子查询 | 逐行判断「存不存在」 | `WHERE EXISTS (SELECT 1 FROM ... WHERE ...)` |
| FROM | 派生表 | 把聚合结果当临时表做两步聚合 | `FROM (SELECT ... GROUP BY ...) AS t` |

### JOIN 心智模型

先问：**要不要保留匹配不上的行？**
- 不要 → **INNER JOIN**（交集）
- 要，保留左表全部 → **LEFT JOIN**（左表为主，右表不匹配填NULL）
- 两边都要 → **FULL JOIN**（并集）

**JOIN 后必做三件事**:
1. 查未匹配行（NULL）→ `COALESCE(col, 'Unknown')` 或 WHERE 排除
2. 确认右表key唯一 → 防止fan-out膨胀
3. 聚合前确认粒度 → `COUNT(*)` vs `COUNT(DISTINCT id)`

**NOT IN 的 NULL 陷阱**: 子查询列可能有NULL时，永远用 `NOT EXISTS`。

## 2. Python 速查卡

### 函数进阶

```python
# 默认参数禁用可变对象
def f(lst=None):
    if lst is None: lst = []

# *args 打包位置参数，**kwargs 打包关键字参数
def g(*args, **kwargs): ...

# 装饰器模板（两件事：签名 + return）
def my_decorator(func):
    def wrapper(*args, **kwargs):
        # 前置逻辑
        result = func(*args, **kwargs)
        # 后置逻辑
        return result  # ← 必须return原函数返回值
    return wrapper
```

### 异常处理黄金法则

| 原则 | 说明 |
|------|------|
| 只抓预期异常 | `except ValueError:` ✓，`except:` ✗（会吞掉真bug） |
| 成功逻辑放 else | `try` 只放可能出错的，成功处理放 `else` |
| 清理放 finally | 资源释放、文件关闭 |
| 自定义异常 | `class MyError(Exception): pass`，让调用方精准捕获 |
| 异常链 | `raise 业务异常 from e`，保留 `e.__cause__` |
| 坏数据跳过+log | `logging.warning(...)` 而非 `print` 或默默丢 |

### 文件 IO

```python
from pathlib import Path
import csv, json, logging

# 读CSV
path = Path("data.csv")
if not path.exists(): return []  # 读前检查存在性
with path.open("r", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))  # 值全是字符串！

# 写CSV（Windows必须 newline=""）
with open("out.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[...])
    writer.writeheader(); writer.writerows(data)

# 写JSON
with open("out.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
```

## 3. NumPy 速查卡

| 操作 | 代码 | 注意 |
|------|------|------|
| 创建 | `np.array([...])`, `np.zeros((2,3))`, `np.arange(10)` | 元素同类型 |
| 向量化 | `arr * 2`, `arr + arr2` | 比for循环快10-100倍 |
| 布尔索引 | `arr[arr > 0]` | 组合条件用 `&` + 括号 |
| axis | `arr.sum(axis=0)` 压行(列汇总) | 记住「被消灭的维度」 |
| 广播 | `matrix * weights.reshape(3,1)` | 用 `keepdims=True` 保持维度 |
| 统计 | `mean/std/max/min/argmax/cumsum/percentile` | `argmax` 返回索引 |
| 占比 | `(arr > 0).mean()` | 布尔数组的mean就是True比例 |
| reshape | `arr.reshape(2, -1)` | `-1` 自动计算 |
| 类型转换 | `arr.astype(float)` | CSV读来的值默认是字符串 |

**量化高频公式**:
- 累计收益率: `np.cumsum(returns)`
- 投资组合日收益: `(returns_matrix * weights.reshape(3,1)).sum(axis=0)`
- Z-score: `(features - mean) / std`

## 4. 综合场景演练 —— 数据分析师的一天

**场景**: 你收到了一份门店销售数据 `sales.csv`，需要完成:
1. 用 Python 读取并清洗数据（类型转换 + 异常处理）
2. 用 NumPy 做统计分析
3. 用 SQL 验证关键指标
4. 把结果输出为 JSON 报告

**数据**: 复用 `../data/sales.csv`（500行，含 order_id, customer_id, product, category, quantity, price, order_date, country, total）

### Step 1: 读取 + 清洗（Python 文件IO + 异常处理）

In [ ]:
import csv
import logging
from pathlib import Path
import numpy as np
import json

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

def load_and_clean(csv_path):
    """读CSV，清洗数据，返回清洗后的dict列表和坏数据记录"""
    path = Path(csv_path)
    if not path.exists():
        logging.warning(f"文件不存在: {path}")
        return []
    
    clean_records = []
    bad_count = 0
    
    with path.open("r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                # 显式类型转换（CSV值全是字符串）
                record = {
                    "order_id": row["order_id"],
                    "customer_id": row["customer_id"],
                    "product": row["product"],
                    "category": row["category"],
                    "quantity": int(row["quantity"]),
                    "price": float(row["price"]),
                    "order_date": row["order_date"],
                    "country": row["country"],
                    "total": float(row["total"])
                }
                clean_records.append(record)
            except (KeyError, ValueError) as e:
                bad_count += 1
                logging.warning(f"跳过坏数据 {row}: {e}")
                continue
    
    logging.info(f"清洗完成: {len(clean_records)} 条有效, {bad_count} 条坏数据")
    return clean_records

records = load_and_clean("../data/sales.csv")
print(f"前3条: {records[:3]}")

### Step 2: NumPy 分析

把清洗后的数据转成 NumPy 数组，做统计分析。

In [ ]:
# 提取 total 列做 NumPy 分析
totals = np.array([r["total"] for r in records])

print(f"订单数: {len(totals)}")
print(f"总销售额: {totals.sum():.2f}")
print(f"平均订单额: {totals.mean():.2f}")
print(f"订单额标准差: {totals.std():.2f}")
print(f"最高订单额: {totals.max():.2f} (索引: {totals.argmax()})")
print(f"中位数: {np.median(totals):.2f}")

# 大订单占比（> 平均额）
big_order_ratio = (totals > totals.mean()).mean()
print(f"大订单占比: {big_order_ratio:.1%}")

### Step 3: SQL 验证（DuckDB）

用 SQL 验证刚才 Python 算出的指标是否一致。

In [ ]:
# 需要 jupysql 或 duckdb 环境
# 以下代码假设你已配置 %%sql magic

"""
%%sql
-- 验证总销售额、平均额、订单数
SELECT
    COUNT(*) AS order_count,
    SUM(total) AS total_sales,
    ROUND(AVG(total), 2) AS avg_order,
    MAX(total) AS max_order
FROM '../data/sales.csv'
"""

"""
%%sql
-- 验证大订单占比（>平均额）
SELECT
    COUNT(*) FILTER (WHERE total > (SELECT AVG(total) FROM '../data/sales.csv')) AS big_orders,
    COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) FILTER (WHERE total > (SELECT AVG(total) FROM '../data/sales.csv')) * 100.0 / COUNT(*),
        1
    ) AS big_order_pct
FROM '../data/sales.csv'
"""

### Step 4: 输出 JSON 报告

In [ ]:
report = {
    "source": "../data/sales.csv",
    "total_records": len(records),
    "total_sales": float(totals.sum()),
    "avg_order": float(totals.mean()),
    "std_order": float(totals.std()),
    "max_order": float(totals.max()),
    "median_order": float(np.median(totals)),
    "big_order_ratio": float(big_order_ratio),
}

output_path = Path("week2_summary.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"报告已写入: {output_path.absolute()}")
print(json.dumps(report, ensure_ascii=False, indent=2))

### 场景总结

这个场景串联了 Week 2 的所有技能:
- **Python 文件IO**: `pathlib` + `csv.DictReader` + `with open`
- **异常处理**: 只抓 `KeyError/ValueError`，坏数据 `logging.warning` + `continue`
- **NumPy**: 数组创建、聚合统计、布尔索引算占比
- **SQL**: 聚合验证、标量子查询做比较
- **JSON输出**: `json.dump` 写报告

**Week 2 核心心法**: 数据岗的工作流 = 读取 → 清洗 → 分析 → 验证 → 报告。每个环节都要异常安全。